In [1]:
""""
Copyright (C) 2025 ETH Zurich, Stefan J.P. Chrin, Gregory Landrum, Sereina Riniker
"""

import os
import pandas as pd
import numpy as np
import glob
import json

# ------ Paths ------
molecules_csv_path = "../../../0_DATA/COMPOUNDS/0_MOLECULES.csv"
base_dir = "./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS"
out_dir = "./B_FORMAT_PREDICTIONS"

# ----- Load molecules CSV -----
molecules_df = pd.read_csv(molecules_csv_path)

predictions = []  # list to store each prediction row
experiment_id = 1  # init experiment counter
prediction_id = 1  # init prediction counter

# ----- Iterate over all PLS_Analysis folders (new file structure) -----
paths = sorted(glob.glob(os.path.join(base_dir, '*', 'PLS_Analysis')))

for path in paths:
    # get target from parent folder
    target = os.path.normpath(path).split(os.sep)[-2]

    # iterate all files in PLS_Analysis
    for file in sorted(os.listdir(path)):
        # ensure file name has 5 underscores
        if file.count('_') != 5:

            raise ValueError(f"File {file} in path {path} does not match expected format (unwanted file in directory).")

        file_path = os.path.join(path, file)
        if os.path.isfile(file_path):
            parts = file.split("_")

            # extract vars from filename
            resolution = parts[3]
            variance_cutoff = parts[5].removesuffix('.json')

            # Load JSON file 
            with open(file_path, 'r') as f:
                data_dict = json.load(f)
                data_predictions = data_dict["split_predictions"]

            # Iterate through each prediction 
            for pred in data_predictions:
                row_dict = pred.copy()
                row_dict.pop("Set", None)  # remove unwanted column
                row_dict["per_target_index"] = row_dict.pop("Per target Index (X)")
                row_dict["target"] = target
                row_dict["resolution"] = resolution
                row_dict["folder"] = file_path
                row_dict["prediction_id"] = prediction_id
                row_dict["variance_cutoff"] = variance_cutoff

                #Lookup molecule info to put in df
                match = molecules_df[
                    (molecules_df['target'] == target) &
                    (molecules_df['per_target_index'] == int(row_dict["per_target_index"]))
                ]

                if match.empty:
                    raise Exception(f"Row not found in molecules CSV: {target} {row_dict['per_target_index']}")
                else:
                    match_row = match.iloc[0]
                    row_dict["smiles"] = match_row.get("smiles")
                    row_dict["molregno"] = match_row.get("molregno")
                    row_dict["ligand"] = match_row.get("ligand")
                    row_dict["sdf_dG_path"] = match_row.get("sdf_dG_path")

                predictions.append(row_dict)
                prediction_id += 1

# Create df from predictions 
predictions_df = pd.DataFrame(predictions)
predictions_df


,split,Actual,Predicted,Residual,per_target_index,target,resolution,folder,prediction_id,variance_cutoff,smiles,molregno,ligand,sdf_dG_path
0,0,-8.35,-8.918840,-0.568840,11,bace,1.0,./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/bace/PLS_Ana...,1,0.001,CN1C(=O)[C@@](c2cncnc2)(c2cccc(-c3cccc(Cl)c3)c...,262,CAT-13m,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-13m.sdf
1,0,-7.44,-7.944263,-0.504263,26,bace,1.0,./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/bace/PLS_Ana...,2,0.001,COc1ccc(-c2cccc([C@@]3(c4ccccc4)[NH+]=C(N)N(C)...,277,CAT-4c,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-4c.sdf
2,0,-9.23,-9.133073,0.096927,7,bace,1.0,./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/bace/PLS_Ana...,3,0.001,CN1C(=O)[C@](c2cccc(-c3cccc(Cl)c3)c2)(C2(C)CC2...,258,CAT-13h,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-13h.sdf
3,0,-8.86,-9.107526,-0.247526,8,bace,1.0,./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/bace/PLS_Ana...,4,0.001,CN1C(=O)[C@@](CC2CC2)(c2cccc(-c3cccc(Cl)c3)c2)...,259,CAT-13i,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-13i.sdf
4,0,-11.35,-9.488940,1.861060,23,bace,1.0,./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/bace/PLS_Ana...,5,0.001,CC#Cc1cncc(-c2cccc([C@@]3(C4CC4)[NH+]=C(N)N(C)...,274,CAT-24,0_DATA/COMPOUNDS/SDF/bace_lig_CAT-24.sdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8895,98,-9.09,-9.843654,-0.753654,7,tyk2,1.0,./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/tyk2/PLS_Ana...,8896,0.001,O=C(Nc1ccnc(NC(=O)C2CCCC2)c1)c1c(Cl)cccc1Cl,322,ejm_48,0_DATA/COMPOUNDS/SDF/tyk2_lig_ejm_48.sdf
8896,99,-11.81,-11.363653,0.446347,12,tyk2,1.0,./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/tyk2/PLS_Ana...,8897,0.001,O=C(Nc1ccnc(NC(=O)[C@H]2C[C@H]2F)c1)c1c(Cl)ccc...,327,jmc_23,0_DATA/COMPOUNDS/SDF/tyk2_lig_jmc_23.sdf
8897,99,-9.63,-10.151742,-0.521742,0,tyk2,1.0,./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/tyk2/PLS_Ana...,8898,0.001,CC(=O)Nc1cc(NC(=O)c2c(Cl)cccc2Cl)ccn1,315,ejm_31,0_DATA/COMPOUNDS/SDF/tyk2_lig_ejm_31.sdf
8898,99,-7.49,-8.839438,-1.349438,3,tyk2,1.0,./A_QSAR_ORIGINAL_PYCOMSIA_FIELDS/tyk2/PLS_Ana...,8899,0.001,CC(C)(C)C(=O)Nc1cc(NC(=O)c2c(Cl)cccc2Cl)ccn1,318,ejm_44,0_DATA/COMPOUNDS/SDF/tyk2_lig_ejm_44.sdf


In [2]:


# ensure split sorted with 0 first
splits_sorted = sorted(predictions_df["split"].unique(), key=lambda x: (x != 0, x))

json_list = []
for split_val in splits_sorted:
    split_df = predictions_df[predictions_df["split"] == split_val]
    split_dict = {}
    for target in sorted(split_df["target"].unique()):
        target_df = split_df[split_df["target"] == target]
        target_dict = {}
        for _, row in target_df.iterrows():
            ligand = f"lig_{row['ligand']}"
            target_dict[ligand] = {
                "prediction": float(row["Predicted"]),
                "experimental": float(row["Actual"])
            }
        split_dict[target] = target_dict
    json_list.append(split_dict)

# save JSON
out_path = os.path.join(out_dir,"dG_pred_&_error_all_splits.json")
with open(out_path, "w") as f:
    json.dump(json_list, f, indent=2)
